In [ ]:
!pip install crewai -q
!pip install langchain -q
!py -m pip install -qU langchain-ollama
!pip install -U ollama
!py -m pip install markdown weasyprint -q
!py -m pip install -qU langchain
!py -m pip install --upgrade langchain langchain-core langchain-community
!py -m pip install langchain-classic

In [ ]:
import pandas as pd
from scipy.stats import boxcox
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.neural_network import MLPRegressor
from pygam import LinearGAM, s
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from pathlib import Path

In [ ]:
def forecast(df):
    X = df[['n_cyanos', 'co2', 'light', 'SucRatio', 'Nsample']]
    y_a = df['a']
    targets = ['a', 'mu', 'tau', 'a0']

    # Split X once
    X_train, X_temp = train_test_split(
        X, test_size=0.3, random_state=42
    )
    X_val, X_test = train_test_split(
        X_temp, test_size=0.5, random_state=42
    )

    # Apply same indices to all targets
    y_a_train, y_a_val, y_a_test = y_a.loc[X_train.index], y_a.loc[X_val.index], y_a.loc[X_test.index]
    X_train_bc = X_train.copy()
    X_val_bc = X_val.copy()
    X_test_bc = X_test.copy()

    boxcox_params = {}  

    for col in X_train.columns:
        min_val = X_train[col].min()
        shift = 0.0
        if min_val <= 0:
            shift = -min_val + 1e-6

        X_train_shifted = X_train[col] + shift
        X_val_shifted = X_val[col] + shift
        X_test_shifted = X_test[col] + shift

        X_train_bc[col], lam = boxcox(X_train_shifted)

        X_val_bc[col] = boxcox(X_val_shifted, lam)
        X_test_bc[col] = boxcox(X_test_shifted, lam)
        
        boxcox_params[col] = {
            "lambda": lam,
            "shift": shift
        }
        xgb = XGBRegressor(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective='reg:squarederror',
        n_jobs=-1,
        random_state=42
    )
    xgb.fit(X_train_bc, y_a_train)

    y_a_train_pred = xgb.predict(X_train_bc)
    y_a_val_pred = xgb.predict(X_val_bc)
    y_a_test_pred = xgb.predict(X_test_bc)
    return {
        "train_r2": r2_score(y_a_train, y_a_train_pred),
        "val_r2": r2_score(y_a_val, y_a_val_pred),
        "test_r2":  r2_score(y_a_test, y_a_test_pred),
        "test_mse": mean_squared_error(y_a_test, y_a_test_pred)
    }

In [20]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "phi4-mini:3.8b" 

In [21]:
@tool
def run_forecast() -> dict:
    """
     Runs the forecasting pipeline and returns model results.
    MUST be called before writing the report.   
    """
    return forecast(df)

tool_result = run_forecast.invoke({})

In [22]:
tool_result

{'train_r2': 0.9997225623579332,
 'val_r2': 0.9724222246266769,
 'test_r2': 0.9748309809789582,
 'test_mse': 4.491240451288654}

In [ ]:
report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert data scientist. "
        "You MUST base your analysis strictly on the provided evaluation results. "
    ),
    (
        "human",
            """
    Here are the forecasting results (JSON):
    {results}

    Write a markdown evaluation report including:
    - Executive summary
    - Metrics table (with exact values)
    - Interpretation
    - Recommendations
    """
    ),
    ])


llm = ChatOllama(
    model=MODEL,
    temperature=0
)

report_chain = report_prompt | llm


In [ ]:
markdown_report = report_chain.invoke({
    "results": tool_result
})

# print(markdown_report.content)
output_path = Path("result/forecast_report.md")
output_path.write_text(markdown_report.content, encoding="utf-8")


# Evaluation Report

## Executive Summary
The forecasting model has demonstrated excellent performance on the training dataset with an R-squared value of 0.9997225623579332, indicating a near-perfect fit to historical data points in this set.

However, there is some drop-off when evaluating against validation and test datasets: 
- The Val R^2 score stands at 0.9724222246266769.
- On the Test dataset (which was not used for training), we achieved an even higher performance with a slightly better fit indicated by its R-squared value of 0.9748309809789582.

The Mean Squared Error on test data is relatively low, at approximately 4.491240451288654 units squared per observation which suggests that the model's predictions are close to actual values but there might still be room for improvement in terms of precision and accuracy across different datasets or under varying conditions not captured during training/testing.

## Metrics Table

| Metric          | Value               |
|-------------

### References
https://docs.langchain.com/oss/python/integrations/chat/ollama
https://docs.langchain.com/oss/python/langgraph/overview

- feed more data about the forecasting process and features for better report


s